# Derivative Trades (D-IFI)

Generates the D-IFI report for derivative trades using the TypeScript/Deno library.

In [ ]:
// ── EDIT THESE VALUES ───────────────────────────────────────────────
import { DateTime } from "luxon";
import { TaxAuthorityLotMatchingMethod, TaxPayerType } from "@brrr/lib";
import type { TaxAuthorityConfiguration, TaxPayerInfo } from "@brrr/lib";

const importsDir = "./imports";    // folder containing IBKR XML exports
const exportsDir = "./exports";    // folder for generated output files

const taxPayerInfo: TaxPayerInfo = {
  taxNumber: "12345678",
  taxpayerType: TaxPayerType.PHYSICAL_SUBJECT,
  name: "Ime Priimek",
  address1: "Ulica 1",
  address2: "",
  city: "Ljubljana",
  postNumber: "1000",
  postName: "Ljubljana",
  municipalityName: "Ljubljana",
  birthDate: DateTime.fromISO("1990-01-01"),
  maticnaStevilka: "",
  invalidskoPodjetje: false,
  resident: true,
  activityCode: "",
  activityName: "",
  countryId: "SI",
  countryName: "Slovenia",
};

const reportConfig: TaxAuthorityConfiguration = {
  fromDate: DateTime.fromISO("2025-01-01"),
  toDate: DateTime.fromISO("2026-01-01"),
  lotMatchingMethod: TaxAuthorityLotMatchingMethod.FIFO,
};

## Load & Process Broker Exports

In [ ]:
import { IbkrBrokerageExportProvider, StagingFinancialGroupingProcessor } from "@brrr/lib";

const ibkrProvider = new IbkrBrokerageExportProvider();

const xmlContents: string[] = [];
for await (const entry of Deno.readDir(importsDir)) {
  if (entry.isFile && entry.name.endsWith(".xml")) {
    xmlContents.push(await Deno.readTextFile(`${importsDir}/${entry.name}`));
  }
}
console.log(`Found ${xmlContents.length} XML file(s)`);

const stagingEvents = ibkrProvider.loadAndTransform(xmlContents);
const groupingProcessor = new StagingFinancialGroupingProcessor();
const financialEvents = groupingProcessor.processStagingFinancialEvents(stagingEvents);

console.log(`Processed ${financialEvents.groupings.length} grouping(s)`);

## Identifier Relationships

In [ ]:
import { ApplyIdentifierRelationshipsService, IdentifierChangeType } from "@brrr/lib";

const applyService = new ApplyIdentifierRelationshipsService();
// Note: SlovenianTaxAuthorityProvider also applies relationships internally.
// This cell is just for inspection.
const inspectedEvents = applyService.apply(financialEvents, [
  IdentifierChangeType.RENAME,
  IdentifierChangeType.SPLIT,
  IdentifierChangeType.REVERSE_SPLIT,
]);

if (inspectedEvents.identifierRelationships.length === 0) {
  console.log("No identifier relationships (no renames/splits found).");
} else {
  for (const rel of inspectedEvents.identifierRelationships) {
    const from = rel.fromIdentifier.getIsin() ?? rel.fromIdentifier.getTicker() ?? "?";
    const to   = rel.toIdentifier.getIsin()   ?? rel.toIdentifier.getTicker()   ?? "?";
    console.log(`${rel.changeType}: ${from} → ${to} (${rel.effectiveDate.toISODate()})`);
  }
}

## Generate D-IFI Report (Derivative Trades)

In [ ]:
import {
  LotMatcher,
  FinancialEventsProcessor,
  CompanyLookupProvider,
  CountryLookupProvider,
  KdvpReportGenerator,
  DivReportGenerator,
  IfiReportGenerator,
  SlovenianTaxAuthorityProvider,
  SlovenianTaxAuthorityReportTypes,
} from "@brrr/lib";

const processor = new FinancialEventsProcessor(null, new LotMatcher());
const provider = new SlovenianTaxAuthorityProvider(
  taxPayerInfo, reportConfig,
  new ApplyIdentifierRelationshipsService(),
  new KdvpReportGenerator(processor),
  new DivReportGenerator(new CompanyLookupProvider(), new CountryLookupProvider()),
  new IfiReportGenerator(processor),
);

// CSV helper
function toCsvString(rows: Record<string, unknown>[]): string {
  if (rows.length === 0) return "";
  const header = Object.keys(rows[0]).join(",");
  const body = rows.map(r => Object.values(r).map(v => `"${String(v ?? "").replace(/"/g, '""')}"`).join(",")).join("\n");
  return header + "\n" + body;
}

const reportType = SlovenianTaxAuthorityReportTypes.D_IFI;

const xmlOutput = provider.generateExportForTaxAuthority(reportType, financialEvents);
const csvRows = provider.generateSpreadsheetExport(reportType, financialEvents);

await Deno.mkdir(exportsDir, { recursive: true });
await Deno.writeTextFile(`${exportsDir}/D_Ifi.xml`, xmlOutput);
await Deno.writeTextFile(`${exportsDir}/export-derivatives.csv`, toCsvString(csvRows));

console.log(`Written: ${exportsDir}/D_Ifi.xml`);
console.log(`Written: ${exportsDir}/export-derivatives.csv (${csvRows.length} row(s))`);